In [1]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque
import matplotlib.pyplot as plt

# 检查 GPU 是否可用
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# grid environment
from environment import WumpusWorldEnvironment

env_grid = WumpusWorldEnvironment(observation_type='integer', action_type='discrete')

obs_sample, _ = env_grid.reset()
#define the input dimension

if isinstance(obs_sample, (int, np.integer)):
    grid_obs_dim = 1 # 或者根据你的地图大小设为格子总数（如果要用One-hot）
else:
    grid_obs_dim = len(obs_sample)

grid_action_dim = env_grid.action_space.n

In [4]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim) # 线性输出，不加 Softmax 或 Sigmoid
        )

    def forward(self, x):
        return self.fc(x)

In [5]:
# Prioritized Replay Buffer

class PrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6):
        self.capacity = capacity
        self.alpha = alpha  # 决定优先级的程度 (0=均匀分布, 1=完全优先级)
        self.buffer = []
        self.pos = 0
        self.priorities = np.zeros((capacity,), dtype=np.float32)
    
    def push(self, state, action, reward, next_state, done):
        # 新样本赋予当前最大优先级，确保它们至少被训练一次
        max_prio = self.priorities.max() if self.buffer else 1.0
        
        if len(self.buffer) < self.capacity:
            self.buffer.append((state, action, reward, next_state, done))
        else:
            self.buffer[self.pos] = (state, action, reward, next_state, done)
        
        self.priorities[self.pos] = max_prio
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size, beta=0.4):
        """
        beta: 采样权重的修正系数 (随着训练增加，逐步从 0.4 增加到 1.0)
        """
        if len(self.buffer) == self.capacity:
            prios = self.priorities
        else:
            prios = self.priorities[:self.pos]
            
        # 根据优先级计算采样概率 P(i) = p^a / sum(p^a)
        probs = prios ** self.alpha
        probs /= probs.sum()

        # 按概率抽取索引
        indices = np.random.choice(len(self.buffer), batch_size, p=probs)
        samples = [self.buffer[idx] for idx in indices]

        # 计算重要性采样权重 (IS weights) 来修正偏差
        # weights = (1/N * 1/P(i))^beta
        total = len(self.buffer)
        weights = (total * probs[indices]) ** (-beta)
        weights /= weights.max()  # 归一化以保持训练稳定
        
        # 解压数据
        states, actions, rewards, next_states, dones = zip(*samples)
        
        return (np.array(states, dtype=np.float32), 
                np.array(actions, dtype=np.int64), 
                np.array(rewards, dtype=np.float32), 
                np.array(next_states, dtype=np.float32), 
                np.array(dones, dtype=np.float32), 
                indices, 
                np.array(weights, dtype=np.float32))

    def update_priorities(self, batch_indices, batch_priorities):
        for idx, prio in zip(batch_indices, batch_priorities):
            # 加上 1e-6 的 epsilon 确保每个样本至少有微小的被选中概率
            self.priorities[idx] = prio + 1e-6

    def __len__(self):
        return len(self.buffer)